# adsb.lol API — Exploration

Direkter HTTP-Zugriff, kein Auth, kein API-Key.  
Base URL: `https://api.adsb.lol/v2/`  

**Hinweis:** Die Doku nennt `/api/v2/` — tatsächlich funktioniert nur `/v2/` (getestet 2026-05-18).

In [ ]:
import requests
import json
import time
import pandas as pd
from datetime import datetime

BASE_URL = "https://api.adsb.lol/v2"

def get(path, **kwargs):
    url = f"{BASE_URL}{path}"
    r = requests.get(url, timeout=10, **kwargs)
    r.raise_for_status()
    return r.json()

print("Setup OK")

## 1. Live-Flugzeuge über Berlin (25 nm Radius)

In [ ]:
data = get("/lat/52.52/lon/13.41/dist/25")

print(f"Zeitstempel:    {datetime.fromtimestamp(data['now'] / 1000)}")
print(f"Flugzeuge:      {data['total']}")
print(f"Server-Zeit:    {data['ptime']} ms")

aircraft = data["ac"]
print(f"\nErste 3 Einträge (roh):")
for a in aircraft[:3]:
    print(json.dumps(a, indent=2))

## 2. Felder analysieren — was ist immer da, was fehlt manchmal?

In [ ]:
df = pd.DataFrame(aircraft)

print(f"Shape: {df.shape}")
print(f"\nSpalten:")
print(df.dtypes.to_string())

print(f"\nFehlende Werte pro Spalte:")
missing = df.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0].to_string())

In [ ]:
# Lesbare Übersicht aller Flugzeuge
cols = [c for c in ["hex", "flight", "r", "t", "alt_baro", "gs", "lat", "lon", "seen"] if c in df.columns]
print(df[cols].to_string(index=False))

## 3. Callsign-Suche — alle Lufthansa-Flüge weltweit

In [ ]:
lh = get("/callsign/DLH")

print(f"Lufthansa-Flüge gerade in der Luft: {lh['total']}")

df_lh = pd.DataFrame(lh["ac"])
cols = [c for c in ["hex", "flight", "r", "t", "alt_baro", "gs", "lat", "lon"] if c in df_lh.columns]
print(df_lh[cols].head(20).to_string(index=False))

## 4. Einzelnes Flugzeug per ICAO24-Hex

In [ ]:
# Ersten Hex aus Berlin-Query nehmen
sample_hex = aircraft[0]["hex"]
print(f"Suche: {sample_hex}")

single = get(f"/hex/{sample_hex}")
print(json.dumps(single["ac"], indent=2) if single["ac"] else "Nicht mehr sichtbar")

## 5. Datenqualität — alt_baro: Zahl vs. 'ground'

In [ ]:
# alt_baro kann int ODER String 'ground' sein — ETL-Fallstrick
if "alt_baro" in df.columns:
    ground = df[df["alt_baro"] == "ground"]
    airborne = df[df["alt_baro"] != "ground"]
    print(f"Am Boden:   {len(ground)}")
    print(f"In der Luft: {len(airborne)}")

    # Für ETL: saubere Normalisierung
    df["alt_baro_ft"] = pd.to_numeric(df["alt_baro"], errors="coerce")  # 'ground' → NaN
    df["on_ground"] = df["alt_baro"] == "ground"
    print(f"\nNach Normalisierung:")
    print(df[["hex", "alt_baro", "alt_baro_ft", "on_ground"]].to_string(index=False))

## 6. Response-Zeit messen — wie schnell ist die API?

In [ ]:
timings = []
for i in range(5):
    t0 = time.time()
    d = get("/lat/52.52/lon/13.41/dist/25")
    elapsed = (time.time() - t0) * 1000
    timings.append(elapsed)
    print(f"Request {i+1}: {elapsed:.0f} ms, {d['total']} aircraft")
    time.sleep(2)

print(f"\nDurchschnitt: {sum(timings)/len(timings):.0f} ms")
print(f"Min/Max:      {min(timings):.0f} / {max(timings):.0f} ms")

## 7. Vergleich: adsb.lol vs. OpenSky — Overlap-Check

Welche Callsigns sehen beide Quellen gerade über Berlin?

In [ ]:
import sys
sys.path.append('.')

# adsb.lol Callsigns über Berlin
adsb_callsigns = set()
for a in aircraft:
    cs = a.get("flight", "").strip()
    if cs:
        adsb_callsigns.add(cs)

print(f"adsb.lol Callsigns über Berlin: {sorted(adsb_callsigns)}")

# OpenSky zum Vergleich (live state vectors über Berlin-Bounding-Box)
try:
    from opensky_api.client import OpenSkyClient
    os_client = OpenSkyClient(use_mock=False)
    states = os_client.get_states(bbox=(51.9, 52.9, 12.9, 14.0))
    os_callsigns = {s.get("callsign", "").strip() for s in states if s.get("callsign")}
    print(f"OpenSky Callsigns über Berlin:  {sorted(os_callsigns)}")
    overlap = adsb_callsigns & os_callsigns
    print(f"\nBeide sehen: {sorted(overlap)}")
    print(f"Nur adsb.lol: {sorted(adsb_callsigns - os_callsigns)}")
    print(f"Nur OpenSky:  {sorted(os_callsigns - adsb_callsigns)}")
except Exception as e:
    print(f"OpenSky nicht verfügbar: {e}")
    print("(Overlap-Check nur mit aktiver DB-Verbindung möglich)")

## 8. Raw-JSON für MongoDB — wie sieht ein Landing-Zone-Dokument aus?

In [ ]:
from datetime import timezone

# Exakt so würde der Collector das Dokument in MongoDB ablegen
landing_doc = {
    "collected_at": datetime.now(timezone.utc).isoformat(),
    "query_type": "lat_lon_dist",
    "query_params": {"lat": 52.52, "lon": 13.41, "dist": 25},
    "source": "adsb.lol",
    "total": data["total"],
    "now": data["now"],
    "ac": data["ac"],
}

print(f"Dokument-Größe: {len(json.dumps(landing_doc))} Bytes")
print(f"Felder top-level: {list(landing_doc.keys())}")
print(f"Aircraft-Einträge: {len(landing_doc['ac'])}")
print(f"\nErwarteter MongoDB-Durchsatz bei 60s-Polling: ~{len(json.dumps(landing_doc)) * 60 // 1024} KB/min")